Import neccessary libraries

In [13]:
import os
import json
import yaml
from pathlib import Path
import logging

import pandas as pd
import optuna
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)

# 1. THIẾT LẬP LOGGING (Dùng force=True để Jupyter không bị lỗi khi chạy lại cell nhiều lần)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True 
)
logger = logging.getLogger(__name__)

# 2. ĐƯỜNG DẪN DỮ LIỆU
CLEANED_DATA_PATH = Path("../data/processed/cleaned_data.csv")
CONFIG_PATH = Path("../artifacts/models/best_model_config.yaml")
TARGET_COLUMN = "churn_status"

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Netflix_All_Models_Tuning")
mlflow.sklearn.autolog()

overall_results = {}

In [3]:
df = pd.read_csv(CLEANED_DATA_PATH)

In [4]:
train_set = df[df['data_split'] == 'train']
test_set = df[df['data_split'] == 'test']

X_train = train_set.drop(columns=['churn_status', 'data_split'])
y_train = train_set['churn_status']

X_test = test_set.drop(columns=['churn_status', 'data_split'])
y_test = test_set['churn_status']

### RANDOM FOREST

In [5]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500), 
        "max_depth": trial.suggest_int("max_depth", 5, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "random_state": 42,
        "class_weight": "balanced",
        "n_jobs": -1 
    }

    with mlflow.start_run(nested=True):
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        metrics = {
            "pr_auc": average_precision_score(y_test, y_proba),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "log_loss": log_loss(y_test, y_proba),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        return metrics["pr_auc"]

with mlflow.start_run(run_name="RandomForest_PR_AUC_Optimization"):
    study_rf = optuna.create_study(direction="maximize")
    study_rf.optimize(objective, n_trials=10) 

    mlflow.log_params(study_rf.best_params)
    mlflow.log_metric("best_pr_auc", study_rf.best_value)

    best_rf = RandomForestClassifier(**study_rf.best_params, random_state=42)
    best_rf.fit(X_train, y_train)
    
    mlflow.sklearn.log_model(best_rf, "best_rf_prauc_model")

    overall_results["RandomForest"] = {
        "model": best_rf,
        "pr_auc": study_rf.best_value,
        "params": study_rf.best_params
    }

    print("-" * 30)
    print(f"RandomForest Xong! Best PR-AUC: {study_rf.best_value:.4f}")
    print(f"Best Parameters: {study_rf.best_params}")

[I 2026-04-05 23:55:17,706] A new study created in memory with name: no-name-1ce50f9e-e90f-4931-ba6b-5350ce829763
2026/04/05 23:55:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-04-05 23:56:09,431] Trial 0 finished with value: 0.23985562794720974 and parameters: {'n_estimators': 316, 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 4, 'criterion': 'entropy'}. Best is trial 0 with value: 0.23985562794720974.


🏃 View run selective-seal-415 at: http://127.0.0.1:5000/#/experiments/4/runs/5cc38c6547f041abaa7b4ec73eee9e53
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/05 23:56:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-04-05 23:56:33,944] Trial 1 finished with value: 0.23869925851417276 and parameters: {'n_estimators': 247, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 1, 'criterion': 'entropy'}. Best is trial 0 with value: 0.23985562794720974.


🏃 View run caring-squirrel-491 at: http://127.0.0.1:5000/#/experiments/4/runs/7451a91bd9db4e188ded4ada4f275b60
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/05 23:56:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-04-05 23:57:20,900] Trial 2 finished with value: 0.24136312741116017 and parameters: {'n_estimators': 144, 'max_depth': 21, 'min_samples_split': 8, 'min_samples_leaf': 2, 'criterion': 'entropy'}. Best is trial 2 with value: 0.24136312741116017.


...

🏃 View run grandiose-stoat-70 at: http://127.0.0.1:5000/#/experiments/4/runs/96bfc621d4254a89a2ec08aec515ac65
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/06 00:00:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-04-06 00:01:15,289] Trial 9 finished with value: 0.2404617070287243 and parameters: {'n_estimators': 446, 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 5, 'criterion': 'gini'}. Best is trial 7 with value: 0.24201202804389005.


🏃 View run persistent-asp-443 at: http://127.0.0.1:5000/#/experiments/4/runs/83932c0365d34a3fb476733f802487ec
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/06 00:01:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
e:\Documents\DSEB.NEU.6TH\ML_OPS\MLOpsProject\MLOpsProject\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2026/04/06 00:01:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 00:01:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution

------------------------------
RandomForest Xong! Best PR-AUC: 0.2420
Best Parameters: {'n_estimators': 335, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 1, 'criterion': 'gini'}
🏃 View run RandomForest_PR_AUC_Optimization at: http://127.0.0.1:5000/#/experiments/4/runs/1067a779a2284fafa278c11890a2a3d3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [6]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Lấy độ quan trọng của các feature từ model tốt nhất
# importances = best_model.feature_importances_
# feature_names = X_train.columns

# # 2. Tạo DataFrame để dễ sắp xếp
# feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# # 3. In ra Top 5
# print("Top 5 Important Features:")
# print(feature_importance_df.head(5))

# # 4. Vẽ biểu đồ trực quan
# plt.figure(figsize=(10, 6))
# plt.barh(feature_importance_df['Feature'].head(5), feature_importance_df['Importance'].head(5), color='skyblue')
# plt.xlabel('Importance Score')
# plt.title('Top 5 Features causing potential Data Leakage')
# plt.gca().invert_yaxis()
# plt.show()

### XGBOOST

In [8]:
ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 5, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, ratio * 1.5),
        "random_state": 42,
        "eval_metric": "logloss"
    }

    with mlflow.start_run(nested=True):
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        metrics = {
            "pr_auc": average_precision_score(y_test, y_proba),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "log_loss": log_loss(y_test, y_proba),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        return metrics["pr_auc"]

with mlflow.start_run(run_name="XGBoost_PR_AUC_Optimization"):
    study_xgb = optuna.create_study(direction="maximize")
    study_xgb.optimize(objective, n_trials=50)

    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metric("best_pr_auc", study_xgb.best_value)

    best_xgb = XGBClassifier(**study_xgb.best_params, random_state=42)
    best_xgb.fit(X_train, y_train)
    
    mlflow.xgboost.log_model(best_xgb, "best_xgb_prauc_model")

    overall_results["XGBoost"] = {
        "model": best_xgb,
        "pr_auc": study_xgb.best_value,
        "params": study_xgb.best_params
    }

    print("-" * 30)
    print(f"XGBoost Xong! Best PR-AUC: {study_xgb.best_value:.4f}")
    print(f"Best Parameters: {study_xgb.best_params}")

[I 2026-04-06 00:02:59,852] A new study created in memory with name: no-name-2c650006-0af0-400d-967f-38ace166ba27
[I 2026-04-06 00:03:08,808] Trial 0 finished with value: 0.22243791492230666 and parameters: {'n_estimators': 957, 'max_depth': 11, 'learning_rate': 0.025407102835048347, 'subsample': 0.7408552841216304, 'colsample_bytree': 0.8290038459512121, 'gamma': 0.22046978139847906, 'scale_pos_weight': 6.886363834460236}. Best is trial 0 with value: 0.22243791492230666.


🏃 View run spiffy-fowl-838 at: http://127.0.0.1:5000/#/experiments/4/runs/f7ba10cb7d4d43758878871d4ed2ffa5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-06 00:03:11,722] Trial 1 finished with value: 0.23472308432043207 and parameters: {'n_estimators': 608, 'max_depth': 7, 'learning_rate': 0.016756988955303535, 'subsample': 0.5237328237754001, 'colsample_bytree': 0.8046665576130911, 'gamma': 0.2596386246155513, 'scale_pos_weight': 6.873499029827826}. Best is trial 1 with value: 0.23472308432043207.


🏃 View run legendary-goat-762 at: http://127.0.0.1:5000/#/experiments/4/runs/8edba00910b6466aa34be8532f790755
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-06 00:03:14,578] Trial 2 finished with value: 0.23667162711245437 and parameters: {'n_estimators': 212, 'max_depth': 13, 'learning_rate': 0.015147533003807018, 'subsample': 0.7450723518025051, 'colsample_bytree': 0.7803560512769854, 'gamma': 3.4865329614922085, 'scale_pos_weight': 6.706154158121369}. Best is trial 2 with value: 0.23667162711245437.


...

🏃 View run painted-crane-468 at: http://127.0.0.1:5000/#/experiments/4/runs/cc9fbb65e2e54064a409bddc6eefbf24
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-06 00:05:07,390] Trial 49 finished with value: 0.23356855867858753 and parameters: {'n_estimators': 570, 'max_depth': 5, 'learning_rate': 0.014537298483094702, 'subsample': 0.6769450403177981, 'colsample_bytree': 0.6902948178978124, 'gamma': 4.100353463663537, 'scale_pos_weight': 4.486437117011097}. Best is trial 15 with value: 0.24003031926686597.


🏃 View run melodic-ant-146 at: http://127.0.0.1:5000/#/experiments/4/runs/31886d1f838a44e9bc8c4c35405519ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/06 00:05:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


------------------------------
XGBoost Xong! Best PR-AUC: 0.2400
Best Parameters: {'n_estimators': 484, 'max_depth': 5, 'learning_rate': 0.012395782189243695, 'subsample': 0.6554059258505803, 'colsample_bytree': 0.7156044404337893, 'gamma': 3.8970724146787483, 'scale_pos_weight': 5.597004317465847}
🏃 View run XGBoost_PR_AUC_Optimization at: http://127.0.0.1:5000/#/experiments/4/runs/f5ea20f3d11d422aa75b8cd0de6a9433
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# # Vẽ ma trận nhầm lẫn chuẩn hóa (normalized) theo hàng ngang (true labels)
# fig, ax = plt.subplots(figsize=(8, 6))
# disp_norm = ConfusionMatrixDisplay.from_estimator(
#     best_xgb, 
#     X_test, 
#     y_test, 
#     display_labels=["Active (0)", "Churn (1)"],
#     cmap="Greens",
#     normalize='true', # Quan trọng: tính % theo từng lớp thực tế
#     ax=ax
# )

# plt.title("Normalized Confusion Matrix (%)")
# plt.show()

### LIGHTGBM

In [9]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, ratio * 1.5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "verbosity": -1
    }

    with mlflow.start_run(nested=True):
        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        metrics = {
            "pr_auc": average_precision_score(y_test, y_proba),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "log_loss": log_loss(y_test, y_proba),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        return metrics["pr_auc"]

with mlflow.start_run(run_name="LGBM_PR_AUC_Optimization"):
    study_lgbm = optuna.create_study(direction="maximize")
    study_lgbm.optimize(objective, n_trials=50)

    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metric("best_pr_auc", study_lgbm.best_value)

    best_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=42)
    best_lgbm.fit(X_train, y_train)
    
    mlflow.lightgbm.log_model(best_lgbm, "best_lgbm_prauc_model")

    overall_results["LightGBM"] = {
        "model": best_lgbm,
        "pr_auc": study_lgbm.best_value,
        "params": study_lgbm.best_params
    }

    print("-" * 30)
    print(f"LGBM Xong! Best PR-AUC: {study_lgbm.best_value:.4f}")
    print(f"Best Parameters: {study_lgbm.best_params}")

[I 2026-04-06 00:07:38,964] A new study created in memory with name: no-name-49c0f6db-bd38-4897-b2ee-302aaef550b6
[I 2026-04-06 00:07:48,643] Trial 0 finished with value: 0.214502060228404 and parameters: {'n_estimators': 731, 'learning_rate': 0.08908715426447054, 'num_leaves': 128, 'max_depth': 11, 'min_child_samples': 46, 'scale_pos_weight': 2.203017303679282, 'reg_alpha': 1.2722233757228854, 'reg_lambda': 0.5477823639332541}. Best is trial 0 with value: 0.214502060228404.


🏃 View run rebellious-yak-802 at: http://127.0.0.1:5000/#/experiments/4/runs/2d2810f813ad4c1d89376ac234d311ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-06 00:08:07,328] Trial 1 finished with value: 0.22863125981442328 and parameters: {'n_estimators': 539, 'learning_rate': 0.029383115261950504, 'num_leaves': 97, 'max_depth': 10, 'min_child_samples': 81, 'scale_pos_weight': 1.5189908512084198, 'reg_alpha': 9.83411201113485, 'reg_lambda': 0.07664358633412773}. Best is trial 1 with value: 0.22863125981442328.


🏃 View run ambitious-deer-125 at: http://127.0.0.1:5000/#/experiments/4/runs/8cb9521b638a40e3ad66100410e4b0ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-06 00:08:23,394] Trial 2 finished with value: 0.21888785444098735 and parameters: {'n_estimators': 1132, 'learning_rate': 0.02410070699281142, 'num_leaves': 120, 'max_depth': 12, 'min_child_samples': 81, 'scale_pos_weight': 2.093371120283229, 'reg_alpha': 0.01144820044684375, 'reg_lambda': 0.014899791555427231}. Best is trial 1 with value: 0.22863125981442328.


...

🏃 View run thundering-wolf-1 at: http://127.0.0.1:5000/#/experiments/4/runs/1c44aae26fd349d88423f3d01bbf7f56
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-04-06 00:11:29,430] Trial 49 finished with value: 0.23499155031509888 and parameters: {'n_estimators': 913, 'learning_rate': 0.014781536592047735, 'num_leaves': 123, 'max_depth': 5, 'min_child_samples': 50, 'scale_pos_weight': 6.111525915363899, 'reg_alpha': 0.0034856661175743093, 'reg_lambda': 0.13038454849943137}. Best is trial 25 with value: 0.24110542508579264.


🏃 View run smiling-newt-259 at: http://127.0.0.1:5000/#/experiments/4/runs/71b6d96eec6d42138929453acc629b46
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/06 00:11:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 00:11:31 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


------------------------------
LGBM Xong! Best PR-AUC: 0.2411
Best Parameters: {'n_estimators': 840, 'learning_rate': 0.018174029105570786, 'num_leaves': 99, 'max_depth': 3, 'min_child_samples': 54, 'scale_pos_weight': 6.4307871058245105, 'reg_alpha': 0.008333539982093801, 'reg_lambda': 0.006967421473158486}
🏃 View run LGBM_PR_AUC_Optimization at: http://127.0.0.1:5000/#/experiments/4/runs/7372f5f3043f456b86b936d3972cc2a2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [14]:

best_model_name = max(overall_results, key=lambda x: overall_results[x]["pr_auc"])
best_info = overall_results[best_model_name]

model_config = {
    'project': 'Netflix_Prediction',
    'best_model_overall': best_model_name,
    'metrics': {
        'pr_auc': float(best_info['pr_auc'])
    },
    'hyperparameters': best_info['params']
}

CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.dump(model_config, f, default_flow_style=False, sort_keys=False)

with mlflow.start_run(run_name="SUMMARY_FINAL_RESULT"):
    mlflow.log_metric("final_best_pr_auc", best_info['pr_auc'])
    mlflow.set_tag("winner_model", best_model_name)
    mlflow.log_dict(model_config, "configs/best_model_config.yaml")
    
    if best_model_name == "LightGBM":
        mlflow.lightgbm.log_model(best_info['model'], "deploy_model")
    elif best_model_name == "XGBoost":
        mlflow.xgboost.log_model(best_info['model'], "deploy_model")
    elif best_model_name == "RandomForest":
        mlflow.sklearn.log_model(best_info['model'], "deploy_model")

2026/04/06 00:20:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 00:20:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SUMMARY_FINAL_RESULT at: http://127.0.0.1:5000/#/experiments/4/runs/213b01e7e0ff488fb25fb41c6aa5e754
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
